# Tome 7, Chapter 2: Hilbert Spaces (Python companion)

Exercises first, solutions at the end. Hilbert space geometry on a computer: kernel ridge regression and the kernel trick, Gram–Schmidt in floating point, Gauss–Legendre quadrature, and orbits that fill the torus. Requirements: numpy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P2.1 (★★) CS application: kernel ridge regression.** Sample $n=40$ noisy values $y_i=\sin(3x_i)+0.2\,\varepsilon_i$ at random $x_i\in[-2,2]$. Fit (a) a polynomial of degree $15$ by least squares and (b) the kernel ridge regression of Exercise 2.6 with the Gaussian kernel $K(x,y)=e^{-(x-y)^2/(2\sigma^2)}$. Compare test errors as $\lambda$ and $\sigma$ vary. Then separate two concentric rings of points in the plane with the kernel method, where no line can separate them.

**P2.2 (★) Gram–Schmidt in floating point.** Orthonormalise the monomials $1,x,\dots,x^{n-1}$ sampled at $1000$ points of $[-1,1]$ with the classical Gram–Schmidt process, the modified one (subtract projections one at a time) and a QR factorisation (Householder). Measure the loss of orthogonality $\|Q^TQ-I\|$ as $n$ grows. Compare the resulting vectors with the normalised Legendre polynomials of Proposition 2.8.

**P2.3 (★) Gauss–Legendre quadrature.** The $n$-point Gauss rule integrates exactly all polynomials of degree $2n-1$, using as nodes the zeros of $P_n$. Compute the nodes and weights, and compare the error of the Gauss rule, the trapezoidal rule and Simpson's rule on $\int_{-1}^1e^x\,dx$, $\int_{-1}^1\frac{dx}{1+25x^2}$ and $\int_{-1}^1|x|\,dx$. Explain the results with best approximation in $L^2$.

**P2.4 (★) Orbits that fill the torus.** Compare the points $n(\sqrt2,\sqrt3)$ modulo $1$, $n<N$, with $N$ independent uniform points: estimate the discrepancy $\sup_{a,b}|\frac{\#\{n<N:x_n\in[0,a[\times[0,b[\}}N-ab|$ for $N$ up to $10^5$. Which converges faster? Then check von Neumann's theorem: for $f(x,y)=\cos(4\pi x)+\cos(6\pi y)$ and for the rational vector $\alpha=(\frac12,\frac13)$, what do the averages $\frac1N\sum_{n<N}f(x+n\alpha)$ converge to?

## Solutions

### P2.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
n = 40
x = rng.uniform(-2, 2, n); y = np.sin(3 * x) + 0.2 * rng.standard_normal(n)
xt = np.linspace(-2, 2, 400); yt = np.sin(3 * xt)

V = np.vander(x, 16); coef = np.linalg.lstsq(V, y, rcond=None)[0]
poly = np.vander(xt, 16) @ coef
print(f"degree-15 polynomial: test RMSE {np.sqrt(np.mean((poly - yt) ** 2)):.3f}")
xe = np.linspace(-2.3, 2.3, 400)
print(f"just outside the data, on [-2.3, 2.3]: max |polynomial| = {np.max(np.abs(np.vander(xe, 16) @ coef)):.1f}")

def krr(x, y, xt, sigma, lam):
    K = lambda a, b: np.exp(-(a[:, None] - b[None, :]) ** 2 / (2 * sigma ** 2))
    a = np.linalg.solve(K(x, x) + lam * np.eye(len(x)), y)
    return K(xt, x) @ a
for sigma in [0.05, 0.3, 2.0]:
    for lam in [1e-6, 1e-2, 1.0]:
        err = np.sqrt(np.mean((krr(x, y, xt, sigma, lam) - yt) ** 2))
        print(f"sigma={sigma:4}, lambda={lam:6}: test RMSE {err:.3f}")
print(f"kernel (sigma=0.3, lambda=1) on [-2.3, 2.3]: max |f| = {np.max(np.abs(krr(x, y, xe, 0.3, 1.0))):.2f}")
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, y, 'k.'); ax.plot(xt, yt, 'k--', lw=0.8)
ax.plot(xt, np.clip(poly, -3, 3), label='polynomial, degree 15')
ax.plot(xt, krr(x, y, xt, 0.3, 1e-2), label='Gaussian kernel, sigma=0.3, lambda=0.01')
ax.legend(fontsize=8); ax.set_ylim(-2.5, 2.5); plt.show()

# two rings: classify by the sign of a kernel ridge fit on labels +-1
m = 200
r = np.concatenate([rng.normal(1, 0.1, m), rng.normal(2, 0.1, m)]); t = rng.uniform(0, 2 * np.pi, 2 * m)
P = np.stack([r * np.cos(t), r * np.sin(t)], 1); lab = np.concatenate([-np.ones(m), np.ones(m)])
idx = rng.permutation(2 * m); tr, te = idx[:300], idx[300:]
K2 = lambda A, B: np.exp(-((A[:, None, :] - B[None, :, :]) ** 2).sum(-1) / 2)
a = np.linalg.solve(K2(P[tr], P[tr]) + 1e-2 * np.eye(300), lab[tr])
pred = np.sign(K2(P[te], P[tr]) @ a)
w = np.linalg.lstsq(np.hstack([P[tr], np.ones((300, 1))]), lab[tr], rcond=None)[0]
pred_lin = np.sign(np.hstack([P[te], np.ones((100, 1))]) @ w)
print("rings: kernel accuracy", np.mean(pred == lab[te]), "  linear accuracy", np.mean(pred_lin == lab[te]))

Inside the data range, the polynomial of degree $15$ does as well as the best kernel fits (test errors near $0.12$–$0.15$, for a noise of standard deviation $0.2$): forty well-spread points pin it down. A step outside, it explodes (the maximum of its modulus on $[-2.3,2.3]$ is printed), while the kernel fit decays gently to $0$ there, because every function of the Gaussian RKHS is a superposition of bumps centred near the data. The kernel fit depends on two knobs. A tiny bandwidth $\sigma=0.05$ makes the kernel functions narrow spikes: with almost no regularisation the linear system is nearly singular and the fit is wild (error above $9$), and even with regularisation the fit is the data plus zeros in between. A wide bandwidth $\sigma=2$ makes the space so smooth that $\sin3x$ is reached only with a tiny $\lambda$; with $\lambda=1$ it underfits badly. With $\sigma=0.3$ the regularised fits are good. The regulariser $\lambda\|f\|^2$ is a norm in the RKHS, a smoothness penalty, and the representer theorem turned an infinite-dimensional problem into a $40\times40$ linear system. On the rings, the best line classifies about half the test points correctly, no better than chance, while the Gaussian kernel classifies all of them: in the RKHS, the rings are separated by a hyperplane.

### P2.2

In [ ]:
import numpy as np
from numpy.polynomial import legendre as Lg
x = np.linspace(-1, 1, 1000)
w = np.full_like(x, 2 / 1000)                               # quadrature weights, so <f,g> ~ int_{-1}^1 f g
def cgs(A):
    Q = np.zeros_like(A)
    for j in range(A.shape[1]):
        v = A[:, j] - Q[:, :j] @ (Q[:, :j].T @ (w * A[:, j]))
        Q[:, j] = v / np.sqrt(np.sum(w * v * v))
    return Q
def mgs(A):
    Q = A.copy().astype(float)
    for j in range(A.shape[1]):
        Q[:, j] /= np.sqrt(np.sum(w * Q[:, j] ** 2))
        for k in range(j + 1, A.shape[1]):
            Q[:, k] -= np.sum(w * Q[:, j] * Q[:, k]) * Q[:, j]
    return Q
for n in [5, 10, 15, 20, 25]:
    A = np.vander(x, n, increasing=True)
    errs = []
    for Q in [cgs(A), mgs(A)]:
        errs.append(np.linalg.norm(Q.T @ (w[:, None] * Q) - np.eye(n)))
    Qh, _ = np.linalg.qr(np.sqrt(w)[:, None] * A); Qh = Qh / np.sqrt(w)[:, None]
    errs.append(np.linalg.norm(Qh.T @ (w[:, None] * Qh) - np.eye(n)))
    print(f"n={n:2d}: ||Q^T Q - I||  classical {errs[0]:.1e}   modified {errs[1]:.1e}   Householder {errs[2]:.1e}")
Q = mgs(np.vander(x, 6, increasing=True))
for k in range(6):
    Pk = Lg.legval(x, [0] * k + [1]) * np.sqrt(k + 0.5)
    print(f"k={k}: max |q_k - sqrt(k+1/2) P_k| = {np.max(np.abs(Q[:, k] - Pk)):.2e}")

In exact arithmetic the three methods agree. In floating point, classical Gram–Schmidt loses orthogonality completely by $n\approx15$: the monomials are nearly parallel on $[-1,1]$ (the Gram matrix is a Hilbert-like matrix, with condition number growing exponentially), and subtracting all projections computed from the *original* vector magnifies rounding errors. Modified Gram–Schmidt, which subtracts each projection from the *current* vector, degrades much more slowly, in proportion to the condition number rather than its square; Householder reflections stay orthogonal to machine precision regardless. For small $n$ the computed vectors are, up to the error of the crude Riemann-sum inner product used here (from $10^{-3}$ for $k=1$ to a few percent for $k=5$, shrinking as the grid is refined), the normalised Legendre polynomials $\sqrt{k+\frac12}P_k$, as Proposition 2.8 predicts: Gram–Schmidt applied to the monomials *is* the Legendre family.

### P2.3

In [ ]:
import numpy as np
from numpy.polynomial import legendre as Lg
fs = {"e^x": (np.exp, np.e - 1 / np.e), "1/(1+25x^2)": (lambda t: 1 / (1 + 25 * t ** 2), 2 * np.arctan(5) / 5), "|x|": (np.abs, 1.0)}
def trap(f, n):
    t = np.linspace(-1, 1, n); return np.trapezoid(f(t), t)
def simpson(f, n):
    n = n if n % 2 else n + 1; t = np.linspace(-1, 1, n); h = t[1] - t[0]
    return h / 3 * (f(t[0]) + f(t[-1]) + 4 * f(t[1:-1:2]).sum() + 2 * f(t[2:-1:2]).sum())
for name, (f, exact) in fs.items():
    print(name)
    for n in [4, 8, 16, 32]:
        xg, wg = Lg.leggauss(n)
        print(f"   n={n:2d}: Gauss {abs(wg @ f(xg) - exact):.1e}   trapezoid {abs(trap(f, n) - exact):.1e}   Simpson {abs(simpson(f, n) - exact):.1e}")
xg, wg = Lg.leggauss(5)
print("5-point nodes:", np.round(xg, 6), " weights:", np.round(wg, 6))
print("exact on x^9?", np.isclose(wg @ xg ** 8, 2 / 9), " on x^10?", np.isclose(wg @ xg ** 10, 2 / 11))

For $e^x$, the Gauss rule with $8$ points is exact to machine precision, while the trapezoid rule with $32$ points still errs by about $10^{-3}$ and Simpson by about $10^{-7}$. Why: an $n$-point Gauss rule is exact on polynomials of degree $2n-1$ (divide $p$ by $P_n$: $p=qP_n+r$ with $\deg q,\deg r<n$; $\int qP_n=0$ by orthogonality, and $r$ is integrated exactly by the interpolatory weights, while $qP_n$ vanishes at the nodes). So its error is controlled by how well $f$ is approximated by polynomials of degree $2n-1$, which for an entire function is super-exponentially well. For the Runge function $\frac1{1+25x^2}$, with poles at $\pm\frac i5$ close to $[-1,1]$, the convergence is still geometric but slow; for $|x|$, with a corner, the Gauss and trapezoid rules converge only like $n^{-2}$, and Gauss loses its edge. (Simpson's rule is exact here by luck: with an odd number of equally spaced nodes, $0$ is a node, and $|x|$ is a polynomial on each half.) The speed of quadrature is the speed of best polynomial approximation, a theme that returns with Chebyshev in the notebooks of Tome C5.

### P2.4

In [ ]:
import numpy as np
rng = np.random.default_rng(4)
def discrepancy(P, grid=64):
    # sup over boxes [0,a)x[0,b) with a, b on a grid (a lower bound for the true star discrepancy)
    H, _, _ = np.histogram2d(P[:, 0], P[:, 1], bins=grid, range=[[0, 1], [0, 1]])
    C = H.cumsum(0).cumsum(1) / len(P)
    a = np.arange(1, grid + 1) / grid
    return np.max(np.abs(C - np.outer(a, a)))
alpha = np.array([np.sqrt(2), np.sqrt(3)])
for N in [100, 1000, 10_000, 100_000]:
    K = (np.arange(N)[:, None] * alpha) % 1
    U = rng.random((N, 2))
    print(f"N={N:6d}: Kronecker {discrepancy(K):.5f}   random {discrepancy(U):.5f}   1/sqrt(N) = {1 / np.sqrt(N):.5f}")
f = lambda P: np.cos(4 * np.pi * P[:, 0]) + np.cos(6 * np.pi * P[:, 1])
x0 = np.array([0.1, 0.2])
for alpha2, name in [(alpha, "(sqrt2, sqrt3)"), (np.array([1 / 2, 1 / 3]), "(1/2, 1/3)")]:
    N = 60_000
    pts = (x0 + np.arange(N)[:, None] * alpha2) % 1
    print(f"alpha={name}: average of f along the orbit = {f(pts).mean():+.5f}   (integral of f = 0)")
print("f averaged over the 6 points x0 + k(1/2,1/3):", f((x0 + np.arange(6)[:, None] * np.array([1 / 2, 1 / 3])) % 1).mean())

Random points have discrepancy of order $\frac1{\sqrt N}$ (the central limit theorem in action), while the Kronecker sequence does much better, roughly like $\frac{\log N}N$ up to constants: it fills the torus *more evenly than chance*. This is the principle of quasi-Monte Carlo integration, which replaces random samples by such sequences (Weyl, Halton, Sobol) and gains a factor of nearly $\sqrt N$ in accuracy for smooth integrands. For the rational vector $(\frac12,\frac13)$, the orbit is periodic with $6$ points, and the averages converge, as von Neumann's theorem promises, but not to $\int f=0$: they converge to the projection of $f$ on the functions invariant under the rotation, which here is the average of $f$ over the $6$-point orbit of $x_0$, about $-0.5$ (the printed value), not $0$. Ergodicity is exactly the property that makes this projection a constant.